# Pokemon Canonical Type Analysis

This notebook rebuilds the type-prediction project around a canonical hybrid dataset:

- `PokeAPI` provides the structured backbone.
- `pokemon.com` official references are attached as URLs and validation metadata.
- Kaggle tables are retained only as historical baseline inputs.

The modeling workflow has three tracks:

1. Historical legacy baseline
2. Clean structured benchmark
3. Structured + text mainline and multimodal image experiment


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from canonical_pokemon import CANONICAL_TABLE_PATHS, ensure_canonical_tables
from pokemon_project import notebook_ready_tables, train_project_bundle

ROOT = Path.cwd()
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)


## Part 1. Why Replace Kaggle

The old project depended on third-party CSV snapshots. That created two problems:

- Some rows were incorrect or stale.
- Many rich fields were missing, especially text, form metadata, provenance, and media links.

This rebuild treats Kaggle as a historical baseline only. The canonical dataset is synchronized from `PokeAPI`, then enriched with official reference URLs and validation metadata.


In [2]:
canonical_tables = ensure_canonical_tables(ROOT, force_refresh=False)
bundle = train_project_bundle(ROOT, include_external=False)

display(bundle["local_data_summary"])
display(bundle["matchup_audit"])


[type] holdout split: random
[type] fitting Structured OVR Logistic on random
[type] fitting Structured ClassifierChain Logistic (C=10) on random


[type] fitting Structured ExtraTrees on random


[type] fitting Structured + Text OVR Logistic on random


[type] fitting Structured + Text ClassifierChain Logistic (C=10) on random


[type] fitting Multimodal Logistic Regression on random


...

[type] evolution-group OOF: Structured + Text ClassifierChain Logistic (C=10)


[type] evolution-group OOF: Multimodal Logistic Regression


[type] evolution-group OOF: Multimodal ExtraTrees


,dataset,rows,columns,missing_cells
0,legacy_cn,1216,11,0
1,pokemon,1215,30,7727
2,single_combats,50000,3,0
3,team_combats,10000,3,0
4,team_ids,100,7,0
5,type_matchup,540,20,0


,original_single_combats_rows,rows_after_old_incomplete_matchup_join,rows_lost_by_old_join,coverage_ratio,unique_pokemon_ids_in_combats,unique_ids_covered_by_matchup_csv
0,50000,15819,34181,0.31638,784,441


## Part 2. Canonical Dataset Inventory

The project now produces normalized entity tables and a denormalized `pokemon_master` table.


In [3]:
inventory_rows = []
for name, path in CANONICAL_TABLE_PATHS.items():
    df = canonical_tables[name]
    inventory_rows.append({"table": name, "rows": len(df), "columns": len(df.columns), "path": str(path)})
inventory_df = pd.DataFrame(inventory_rows).sort_values("table").reset_index(drop=True)
display(inventory_df)

display(canonical_tables["pokemon_master"].head(8))


,table,rows,columns,path
0,ability_details,310,4,/Users/zhangziling/Documents/MSIS/MSIS510/Presentation/artifacts/canonical/ability_details.csv
1,move_details,833,7,/Users/zhangziling/Documents/MSIS/MSIS510/Presentation/artifacts/canonical/move_details.csv
2,official_validation_report,1350,10,/Users/zhangziling/Documents/MSIS/MSIS510/Presentation/artifacts/canonical/official_validation_report.csv
3,pokemon,1350,54,/Users/zhangziling/Documents/MSIS/MSIS510/Presentation/artifacts/canonical/pokemon.csv
4,pokemon_abilities,2926,6,/Users/zhangziling/Documents/MSIS/MSIS510/Presentation/artifacts/canonical/pokemon_abilities.csv
5,pokemon_evolution_edges,484,11,/Users/zhangziling/Documents/MSIS/MSIS510/Presentation/artifacts/canonical/pokemon_evolution_edges.csv
6,pokemon_flavor_texts,8490,4,/Users/zhangziling/Documents/MSIS/MSIS510/Presentation/artifacts/canonical/pokemon_flavor_texts.csv
7,pokemon_forms,1350,11,/Users/zhangziling/Documents/MSIS/MSIS510/Presentation/artifacts/canonical/pokemon_forms.csv
8,pokemon_master,1350,92,/Users/zhangziling/Documents/MSIS/MSIS510/Presentation/artifacts/canonical/pokemon_master.csv
9,pokemon_media,1350,14,/Users/zhangziling/Documents/MSIS/MSIS510/Presentation/artifacts/canonical/pokemon_media_manifest.csv


,pokemon_id,pokemon_name,canonical_slug,display_name,species_slug,dexnum_int,order,is_default,base_experience,height,weight,type1,type2,hp,attack,defense,sp_atk,sp_def,speed,total,generation,capture_rate,base_happiness,base_exp,growth_rate,egg_group1,egg_group2,percent_male,percent_female,egg_cycles,color,shape,habitat,is_baby,is_legendary,is_mythical,region_tag,form_group,is_mega,is_battle_only,evolution_chain_id,evolution_stage,genus_en,flavor_text_en,flavor_text_corpus_en,ability1,ability2,hidden_ability,ability_summary_en,official_artwork_url,artwork_variant_url,sprite_url,image_url,official_pokedex_url,species_display_name,form_display_name,text_corpus_en,validation_status,validated_at,mismatch_fields,move_type_count_Bug,move_type_count_Dark,move_type_count_Dragon,move_type_count_Electric,move_type_count_Fairy,move_type_count_Fighting,move_type_count_Fire,move_type_count_Flying,move_type_count_Ghost,move_type_count_Grass,move_type_count_Ground,move_type_count_Ice,move_type_count_Normal,move_type_count_Poison,move_type_count_Psychic,move_type_count_Rock,move_type_count_Steel,move_type_count_Water,move_class_count_physical,move_class_count_special,move_class_count_status,move_count,single_type_flag,bulk_score,offense_score,physical_bias,special_bias,speed_rank_pct,special_group,pokemon_api_ability_count,base_stat_total_check,total_matches_sum
0,1,bulbasaur,bulbasaur,Bulbasaur,bulbasaur,1,1,1,64.0,7,69,Grass,Poison,45,49,49,65,65,45,318,1,45,70,64.0,Medium Slow,Monster,Plant,87.5,12.5,20,Green,Quadruped,Grassland,0,0,0,NaN,Standard,0,0,1,0,Seed Pokémon,A strange seed was planted on its back at birth. The plant sprouts and grows with this POKéMON.,A strange seed was planted on its back at birth. The plant sprouts and grows with this POKéMON. It can go for days w...,Overgrow,Unknown,Chlorophyll,Strengthens grass moves to inflict 1.5× damage at 1/3 max HP or less. Doubles Speed during strong sunlight.,https://assets.pokemon.com/assets/cms2/img/pokedex/full/001.png,https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/1.png,https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/home/1.png,https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/1.png,https://www.pokemon.com/us/pokedex/bulbasaur,Bulbasaur,Default Form,Bulbasaur Bulbasaur Seed Pokémon A strange seed was planted on its back at birth. The plant sprouts and grows with t...,official_media_verified_link_generated,2026-04-13T11:45:21.940891-07:00,NaN,2.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,23.0,1.0,0.0,44.0,6.0,4.0,0.0,0.0,0.0,28.0,21.0,37.0,86.0,0,159,159,-16,16,0.220370,Ordinary,2,318,1
1,2,ivysaur,ivysaur,Ivysaur,ivysaur,2,2,1,142.0,10,130,Grass,Poison,60,62,63,80,80,60,405,1,45,70,142.0,Medium Slow,Monster,Plant,87.5,12.5,20,Green,Quadruped,Grassland,0,0,0,NaN,Standard,0,0,1,1,Seed Pokémon,"When the bulb on its back grows large, it appears to lose the ability to stand on its hind legs.","When the bulb on its back grows large, it appears to lose the ability to stand on its hind legs. The bulb on its bac...",Overgrow,Unknown,Chlorophyll,Strengthens grass moves to inflict 1.5× damage at 1/3 max HP or less. Doubles Speed during strong sunlight.,https://assets.pokemon.com/assets/cms2/img/pokedex/full/002.png,https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/2.png,https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/home/2.png,https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/other/official-artwork/2.png,https://www.pokemon.com/us/pokedex/ivysaur,Ivysaur,Default Form,"Ivysaur Ivysaur Seed Pokémon When the bulb on its back grows large, it appears to lose the ability to stand on its h...",official_media_verified_link_generated,2026-04-13T11:45:21.940891-07:00,NaN,2.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,22.0,1.0,0.0,43.0,5.0,4.0,0.0,0.0,0.0,27.0,19.0,37.0,83.0,0,203,202,-18,18,0.379630,

## Part 3. Validation and Provenance

Because `pokemon.com` blocks large-scale automated page scraping, this pipeline uses a conservative official-reference strategy:

- attach official Pokédex URLs
- verify official artwork URLs where possible
- record validation status and manual-review notes


In [4]:
validation_df = bundle["official_validation_report"].copy()
coverage = pd.DataFrame(
    [
        {
            "rows": len(validation_df),
            "official_artwork_verified_pct": validation_df["official_artwork_http_status"].eq(200).mean(),
            "manual_review_required_pct": validation_df["validation_status"].eq("manual_review_required").mean(),
        }
    ]
)
display(coverage)
display(validation_df.head(10))


,rows,official_artwork_verified_pct,manual_review_required_pct
0,1350,1.0,0.0


,canonical_slug,display_name,dexnum_int,official_pokedex_url,official_artwork_url,official_artwork_http_status,validation_status,validated_at,mismatch_fields,validation_notes
0,bulbasaur,Bulbasaur,1,https://www.pokemon.com/us/pokedex/bulbasaur,https://assets.pokemon.com/assets/cms2/img/pokedex/full/001.png,200,official_media_verified_link_generated,2026-04-13T11:45:21.940891-07:00,NaN,pokemon.com page content is bot-protected; official page URL is attached for manual verification.
1,ivysaur,Ivysaur,2,https://www.pokemon.com/us/pokedex/ivysaur,https://assets.pokemon.com/assets/cms2/img/pokedex/full/002.png,200,official_media_verified_link_generated,2026-04-13T11:45:21.940891-07:00,NaN,pokemon.com page content is bot-protected; official page URL is attached for manual verification.
2,venusaur,Venusaur,3,https://www.pokemon.com/us/pokedex/venusaur,https://assets.pokemon.com/assets/cms2/img/pokedex/full/003.png,200,official_media_verified_link_generated,2026-04-13T11:45:21.940891-07:00,NaN,pokemon.com page content is bot-protected; official page URL is attached for manual verification.
3,venusaur-gmax,Venusaur Gmax,3,https://www.pokemon.com/us/pokedex/venusaur,https://assets.pokemon.com/assets/cms2/img/pokedex/full/003.png,200,official_media_verified_link_generated,2026-04-13T11:45:21.940891-07:00,NaN,pokemon.com page content is bot-protected; official page URL is attached for manual verification.
4,venusaur-mega,Venusaur maleega,3,https://www.pokemon.com/us/pokedex/venusaur,https://assets.pokemon.com/assets/cms2/img/pokedex/full/003.png,200,official_media_verified_link_generated,2026-04-13T11:45:21.940891-07:00,NaN,pokemon.com page content is bot-protected; official page URL is attached for manual verification.
5,charmander,Charmander,4,https://www.pokemon.com/us/pokedex/charmander,https://assets.pokemon.com/assets/cms2/img/pokedex/full/004.png,200,official_media_verified_link_generated,2026-04-13T11:45:21.940891-07:00,NaN,pokemon.com page content is bot-protected; official page URL is attached for manual verification.
6,charmeleon,Charmeleon,5,https://www.pokemon.com/us/pokedex/charmeleon,https://assets.pokemon.com/assets/cms2/img/pokedex/full/005.png,200,official_media_verified_link_generated,2026-04-13T11:45:21.940891-07:00,NaN,pokemon.com page content is bot-protected; official page URL is attached for manual verification.
7,charizard,Charizard,6,https://www.pokemon.com/us/pokedex/charizard,https://assets.pokemon.com/assets/cms2/img/pokedex/full/006.png,200,official_media_verified_link_generated,2026-04-13T11:45:21.940891-07:00,NaN,pokemon.com page content is bot-protected; official page URL is attached for manual verification.
8,charizard-gmax,Charizard Gmax,6,https://www.pokemon.com/us/pokedex/charizard,https://assets.pokemon.com/assets/cms2/img/pokedex/full/006.png,200,official_media_verified_link_generated,2026-04-13T11:45:21.940891-07:00,NaN,pokemon.com page content is bot-protected; official page URL is attached for manual verification.
9,charizard-mega-x,Charizard maleega X,6,https://www.pokemon.com/us/pokedex/charizard,https://assets.pokemon.com/assets/cms2/img/pokedex/full/006.png,200,official_media_verified_link_generated,2026-04-13T11:45:21.940891-07:00,NaN,pokemon.com page content is bot-protected; official page URL is attached for manual verification.


## Part 4. Historical Baseline vs Canonical Benchmark

The next table compares the old CSV-style baseline against the new canonical pipeline.


In [5]:
display(bundle["legacy_type_reports"])
display(bundle["type_reports"])
display(bundle["type_selection"])


,task,model,split,micro_f1,macro_f1,hamming_loss,exact_match,train_rows,test_rows
0,legacy_type_prediction,Legacy ClassifierChain Logistic (C=10),legacy_random,0.734641,0.684896,0.052118,0.487805,820,205
1,legacy_type_prediction,Legacy ExtraTrees,legacy_random,0.698830,0.629268,0.052888,0.346341,820,205
2,legacy_type_prediction,Legacy OVR Logistic,legacy_random,0.631259,0.527819,0.062388,0.243902,820,205


,task,model,feature_mode,split,micro_f1,macro_f1,hamming_loss,exact_match,train_rows,test_rows
0,type_prediction,Structured + Text ClassifierChain Logistic (C=10),structured_text,random,0.911848,0.889105,0.018129,0.762963,1080,270
1,type_prediction,Structured ClassifierChain Logistic (C=10),structured,random,0.909953,0.888075,0.018519,0.759259,1080,270
2,type_prediction,Structured + Text OVR Logistic,structured_text,random,0.903475,0.879794,0.019493,0.718519,1080,270
3,type_prediction,Structured OVR Logistic,structured,random,0.903475,0.879528,0.019493,0.714815,1080,270
4,type_prediction,Structured ExtraTrees,structured,random,0.842105,0.775243,0.029240,0.574074,1080,270
5,type_prediction,Multimodal Logistic Regression,multimodal,random,0.838839,0.792828,0.031384,0.540741,1080,270
6,type_prediction,Multimodal ExtraTrees,multimodal,random,0.652658,0.489736,0.054776,0.274074,1080,270
7,type_prediction,Structured ClassifierChain Logistic (C=10),structured,species_group,0.877295,0.835947,0.025319,0.685606,1086,264
8,type_prediction,Structured + Text ClassifierChain Logistic (C=10),structured_text,species_group,0.872340,0.830309,0.026316,0.670455,1086,264
9,type_prediction,Structured + Text OVR Logistic,structured_text,species_group,0.872763,0.830525,0.025518,0.647727,1086,264


,model,feature_mode,split,micro_f1,macro_f1,exact_match,all_types_correct_n,one_type_correct_n,zero_type_correct_n,n_total,ordered_match_n,deploy_priority
0,Structured + Text ClassifierChain Logistic (C=10),structured_text,evolution_group_oof,0.816667,0.817299,0.665926,899,350,101,1350,700,1
1,Structured OVR Logistic,structured,evolution_group_oof,0.811481,0.817782,0.662963,895,343,112,1350,697,2
2,Structured + Text OVR Logistic,structured_text,evolution_group_oof,0.811111,0.817865,0.661481,893,346,111,1350,698,2
3,Structured ClassifierChain Logistic (C=10),structured,evolution_group_oof,0.812222,0.811438,0.660000,891,355,104,1350,693,1
4,Multimodal Logistic Regression,multimodal,evolution_group_oof,0.744444,0.750944,0.541481,731,464,155,1350,592,4
5,Structured ExtraTrees,structured,evolution_group_oof,0.720741,0.713981,0.493333,666,523,161,1350,549,3
6,Multimodal ExtraTrees,multimodal,evolution_group_oof,0.686296,0.670695,0.430370,581,587,182,1350,527,5


## Part 5. Out-of-Fold Benchmark Summary

Formal reporting should come from evolution-group OOF, not from training-set predictions.


In [6]:
display(bundle["type_oof_summary"])

evolution_oof = bundle["type_oof_summary"].query("split_mode == 'evolution_group'").copy()
display(evolution_oof)


,split_mode,model,feature_mode,n_splits,n_total,all_types_correct_n,all_types_correct_pct,one_type_correct_n,one_type_correct_pct,zero_type_correct_n,zero_type_correct_pct,ordered_match_n,ordered_match_pct
0,random,Structured + Text ClassifierChain Logistic (C=10),structured_text,5,1350,1089,0.806667,202,0.149630,59,0.043704,860,0.637037
1,species_group,Structured + Text ClassifierChain Logistic (C=10),structured_text,5,1350,1032,0.764444,238,0.176296,80,0.059259,816,0.604444
2,evolution_group,Structured + Text ClassifierChain Logistic (C=10),structured_text,5,1350,899,0.665926,350,0.259259,101,0.074815,700,0.518519


,split_mode,model,feature_mode,n_splits,n_total,all_types_correct_n,all_types_correct_pct,one_type_correct_n,one_type_correct_pct,zero_type_correct_n,zero_type_correct_pct,ordered_match_n,ordered_match_pct
2,evolution_group,Structured + Text ClassifierChain Logistic (C=10),structured_text,5,1350,899,0.665926,350,0.259259,101,0.074815,700,0.518519


## Part 6. Random Split vs Species Group vs Evolution Group

- `random split` is historically comparable but optimistic.
- `species group` is stricter because it prevents same-species leakage.
- `evolution group OOF` is the main benchmark because it also blocks same-family leakage.

This is not LLM-style hallucination. When the model is wrong here, it is a generalization error under a stricter holdout definition.


In [7]:
oof_summary = bundle["type_oof_summary"].copy()
comparison = oof_summary[[
    "split_mode",
    "all_types_correct_n",
    "all_types_correct_pct",
    "one_type_correct_n",
    "one_type_correct_pct",
    "zero_type_correct_n",
    "zero_type_correct_pct",
]]
display(comparison)


,split_mode,all_types_correct_n,all_types_correct_pct,one_type_correct_n,one_type_correct_pct,zero_type_correct_n,zero_type_correct_pct
0,random,1089,0.806667,202,0.149630,59,0.043704
1,species_group,1032,0.764444,238,0.176296,80,0.059259
2,evolution_group,899,0.665926,350,0.259259,101,0.074815


## Part 7. Case Studies

These are examples that connect back to the original motivation: Pokemon that look like one type to humans, but are actually something else.


In [8]:
display(bundle["case_study_results"])

detail_df = bundle["type_oof_details"]
final_model_name = bundle["type_bundle"]["final_model_name"]
mistakes = detail_df[(detail_df["model"] == final_model_name) & (detail_df["result_bucket"] != "all_types_correct")].copy()
display(mistakes.head(20))


,canonical_slug,display_name,type1,type2,species_display_name,official_pokedex_url,validation_status,predicted_primary,predicted_secondary,matched_type_count,result_bucket
0,charizard,Charizard,Fire,Flying,Charizard,https://www.pokemon.com/us/pokedex/charizard,official_media_verified_link_generated,Fire,Dragon,1,one_type_correct
1,exeggutor-alola,Exeggutor Alola,Grass,Dragon,Exeggutor,https://www.pokemon.com/us/pokedex/exeggutor,official_media_verified_link_generated,Grass,Psychic,1,one_type_correct
2,gyarados,Gyarados,Water,Flying,Gyarados,https://www.pokemon.com/us/pokedex/gyarados,official_media_verified_link_generated,Dark,Water,1,one_type_correct
3,lugia,Lugia,Psychic,Flying,Lugia,https://www.pokemon.com/us/pokedex/lugia,official_media_verified_link_generated,Flying,None,1,one_type_correct
4,goodra-hisui,Goodra Hisui,Steel,Dragon,Goodra,https://www.pokemon.com/us/pokedex/goodra,official_media_verified_link_generated,Dragon,Poison,1,one_type_correct


,model,feature_mode,fold,canonical_slug,display_name,species_slug,evolution_chain_id,true_primary,true_secondary,predicted_primary,predicted_secondary,matched_type_count,result_bucket,ordered_match
2700,Structured + Text ClassifierChain Logistic (C=10),structured_text,1,absol,Absol,absol,185,Dark,None,Dark,Normal,1,one_type_correct,False
2702,Structured + Text ClassifierChain Logistic (C=10),structured_text,1,absol-mega-z,Absol maleega Z,absol,185,Dark,Ghost,Fighting,None,0,zero_type_correct,False
2703,Structured + Text ClassifierChain Logistic (C=10),structured_text,1,aggron,Aggron,aggron,151,Steel,Rock,Steel,None,1,one_type_correct,False
2711,Structured + Text ClassifierChain Logistic (C=10),structured_text,1,banette-mega,Banette maleega,banette,181,Ghost,None,Ghost,Dark,1,one_type_correct,False
2713,Structured + Text ClassifierChain Logistic (C=10),structured_text,1,barbaracle-mega,Barbaracle maleega,barbaracle,353,Rock,Fighting,Fighting,None,1,one_type_correct,False
2717,Structured + Text ClassifierChain Logistic (C=10),structured_text,1,binacle,Binacle,binacle,353,Rock,Water,Water,None,1,one_type_correct,False
2721,Structured + Text ClassifierChain Logistic (C=10),structured_text,1,braviary-hisui,Braviary Hisui,braviary,319,Psychic,Flying,Flying,Normal,1,one_type_correct,False
2726,Structured + Text ClassifierChain Logistic (C=10),structured_text,1,butterfree-gmax,Butterfree Gmax,butterfree,4,Bug,Flying,Poison,Bug,1,one_type_correct,False
2731,Structured + Text ClassifierChain Logistic (C=10),structured_text,1,centiskorch-gmax,Centiskorch Gmax,centiskorch,447,Fire,Bug,Fairy,Dark,0,zero_type_correct,False
2732,Structured + Text ClassifierChain Logistic (C=10),structured_text,1,cetitan,Cetitan,cetitan,511,Ice,None,Ice,Normal,1,one_type_correct,False


## Part 8. Streamlit Deployment Notes

- The Streamlit app uses a prebuilt deploy artifact.
- It clearly separates `Ground Truth` from `Model Prediction`.
- It also shows provenance, validation status, and benchmark-vs-deployment framing.


In [9]:
tables = notebook_ready_tables(bundle)
display(tables["type_random"])
display(tables["type_grouped"])
display(bundle["battle_reports"])


,task,model,feature_mode,split,micro_f1,macro_f1,hamming_loss,exact_match,train_rows,test_rows
0,type_prediction,Structured + Text ClassifierChain Logistic (C=10),structured_text,random,0.911848,0.889105,0.018129,0.762963,1080,270
1,type_prediction,Structured ClassifierChain Logistic (C=10),structured,random,0.909953,0.888075,0.018519,0.759259,1080,270
2,type_prediction,Structured + Text OVR Logistic,structured_text,random,0.903475,0.879794,0.019493,0.718519,1080,270
3,type_prediction,Structured OVR Logistic,structured,random,0.903475,0.879528,0.019493,0.714815,1080,270
4,type_prediction,Structured ExtraTrees,structured,random,0.842105,0.775243,0.029240,0.574074,1080,270
5,type_prediction,Multimodal Logistic Regression,multimodal,random,0.838839,0.792828,0.031384,0.540741,1080,270
6,type_prediction,Multimodal ExtraTrees,multimodal,random,0.652658,0.489736,0.054776,0.274074,1080,270


,task,model,feature_mode,split,micro_f1,macro_f1,hamming_loss,exact_match,train_rows,test_rows
0,type_prediction,Structured ClassifierChain Logistic (C=10),structured,species_group,0.877295,0.835947,0.025319,0.685606,1086,264
1,type_prediction,Structured + Text ClassifierChain Logistic (C=10),structured_text,species_group,0.872340,0.830309,0.026316,0.670455,1086,264
2,type_prediction,Structured + Text OVR Logistic,structured_text,species_group,0.872763,0.830525,0.025518,0.647727,1086,264
3,type_prediction,Structured OVR Logistic,structured,species_group,0.872404,0.831019,0.025718,0.643939,1086,264
4,type_prediction,Multimodal Logistic Regression,multimodal,species_group,0.812371,0.755913,0.036284,0.530303,1086,264
5,type_prediction,Structured ExtraTrees,structured,species_group,0.796875,0.711784,0.036284,0.507576,1086,264
6,type_prediction,Multimodal ExtraTrees,multimodal,species_group,0.596583,0.392235,0.061204,0.246212,1086,264


,task,model,split,feature_set,accuracy,roc_auc,train_rows,test_rows
0,battle_prediction,Extra Trees,grouped,full,0.8018,0.875599,40000,10000
1,battle_prediction,Random Forest,grouped,full,0.7946,0.869895,40000,10000
2,battle_prediction,Logistic Regression,grouped,baseline,0.5377,0.546011,40000,10000
3,battle_prediction,Extra Trees,random,full,0.8153,0.895845,40000,10000
4,battle_prediction,Random Forest,random,full,0.8088,0.892208,40000,10000
5,battle_prediction,Logistic Regression,random,baseline,0.5358,0.534854,40000,10000
